# Reproducing Gao, Ramachandran, Yu & He (2025) from dandiset 000946

Paper. Gao H, Ramachandran S, Yu K, He B (2025). Transcranial focused ultrasound modulates feedforward
and feedback cortico-thalamo-cortical pathways by selectively activating excitatory neurons. *J Neurosci*
45(23): e2218242025. [doi:10.1523/jneurosci.2218-24.2025](https://doi.org/10.1523/jneurosci.2218-24.2025).
The claims file (`claims/000946/10.1523-jneurosci.2218-24.2025.yaml`) was written from the version of
record as served by PubMed Central (PMC12139594): main text, Methods and figure legends; the paper has no
supplement and links no analysis code.

Dandiset. [DANDI:000946](https://dandiarchive.org/dandiset/000946/0.250829.1249), version
0.250829.1249. The paper cites the dandiset without a version (Methods, Electrophysiological recordings:
"All experimental data has been made public in a repository at the DANDI Archive
(https://dandiarchive.org/dandiset/000946)"), so the most recent published version is pinned. The data
check below compares its file list with the earliest published version, 0.250711.1640; all six published
versions list the same 1920 files.

What the dataset holds. One NWB file per recorded area and session, each with a units table (spike
times and a `celltype_label` column) and a trials table. There are no spike waveforms in any file, and
`celltype_label`, described as "label 1 = RSU, label 2 = FSU", is 0 for every unit. The paper splits units
into RSUs and FSUs by k-means on waveform measures (Methods, Neuronal type classification), so every claim
about RSUs or FSUs, and every claim built on them, cannot be tested. No session is marked as sham or as
tFUS aimed at CG1. The auditory cortex units of Figure 11B are not split by type, and that claim is tested.

| Claim | Tested here |
|---|---|
| fig4-unit-counts | Blocked: data missing (no cell types); totals per area shown |
| fig4a-s1-rsu-time-locked-3000 | Blocked: data missing (no cell types) |
| fig4b-pom-rsu-time-locked-3000 | Blocked: data missing (no cell types) |
| fig4-postsonication-3000 | Blocked: data missing (no cell types) |
| fig4-30hz-responses | Blocked: data missing (no cell types) |
| fig4-fsu-no-change | Blocked: data missing (no cell types) |
| fig5a-sham-no-change | Blocked: data missing (no cell types, no sham sessions marked) |
| fig5b-cg1-control-no-change | Blocked: data missing (no cell types, no CG1 sessions marked) |
| fig6-unit-counts | Blocked: data missing (no cell types); totals per area shown |
| fig6a-s1-rsu-time-locked-prf, fig6a-s1-rsu-delayed-prf, fig6b-s1-fsu-prf, fig6c-pom-rsu-prf, fig6d-pom-fsu-prf | Blocked: data missing (no cell types) |
| fig7a to fig7d, fig8a, fig8b, fig9a, fig9b (8 claims) | Blocked: data missing (no cell types) |
| fig10a-dsttc-3000, fig10b-dsttc-30, fig10c to fig10f (6 claims) | Blocked: data missing (no cell types) |
| pathway-follows-local-prf | Blocked: depends on fig6a and fig10c |
| fig11b-ac-responses | Tested |
| fig11c-ac-to-s1-30hz, fig11d-ac-to-s1-3000 | Blocked: data missing (no S1 cell types) |
| auditory-confound-low-prf | Blocked: depends on fig11c and fig11d |

No step of the analysis is random, so no seed is needed.

In [ ]:
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import tfus

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
print("dandiset", tfus.DANDISET_ID, "version", tfus.VERSION)

## Data check

Every file of the pinned version is opened and its metadata, trial count and per-unit spike counts are
read (not the spike times). Each check of the metadata against the values is stated under its output.

In [ ]:
assets = tfus.list_assets()
early = tfus.list_assets("0.250711.1640")
same = {(a["path"], a["size"]) for a in assets} == {(a["path"], a["size"]) for a in early}
print(f"{len(assets)} files in {tfus.VERSION}; {len(early)} in 0.250711.1640; same paths and sizes: {same}")
t0 = time.time()
inv = tfus.inventory(assets)
print(f"read the headers of {len(inv)} files in {time.time() - t0:.0f} s; files that failed to open: "
      f"{(inv.error != '').sum()}")
inv["kind"] = np.where(inv.path.str.startswith("sub-STTC"), "STTC", "units")
print(inv.kind.value_counts().to_string())

In [ ]:
u = inv[inv.kind == "units"].copy()
print("Columns of the units tables:")
print(u.unit_columns.value_counts().to_string())
print("\nNon-empty acquisition, analysis or processing groups (where waveforms could be):",
      int((u.groups != "").sum()), "of", len(u), "files")
print("\nValues of celltype_label over all files (absent where the column is missing):")
print(u.labels.value_counts().to_string())
n_units_all = int(u.n_units.sum())
print(f"\n{n_units_all} units in {len(u)} unit files, before removing duplicate files")

The units tables hold only spike times and `celltype_label`. No file has waveforms, mean waveforms or
anything under acquisition, analysis or processing, and `celltype_label` is 0 in every file where it is
present (it is missing from the rest), though its description says 1 is RSU and 2 is FSU. The RSU/FSU
split of the paper cannot be made from these data.

In [ ]:
kept, dropped = tfus.dedupe(u)
print(f"{len(u)} unit files; {len(dropped)} have the same subject, identifier, area and per-unit spike counts "
      f"as another file; {len(kept)} distinct recordings remain")
print("Files per group of duplicates:", (dropped.kept.value_counts() + 1).value_counts().to_dict())
ids = dict(zip(u.path, u.asset_id))
t0 = time.time()
h_kept = dict(zip(dropped.kept.unique(), tfus.map_sessions(tfus.spike_hash, [ids[p] for p in dropped.kept.unique()])))
h_drop = tfus.map_sessions(tfus.spike_hash, [ids[p] for p in dropped.dropped])
ident = sum(h_kept[k] == h for k, h in zip(dropped.kept, h_drop))
print(f"duplicate files whose spike times and trial starts are identical to the kept file: {ident} of {len(dropped)} "
      f"({time.time() - t0:.0f} s)")
st = u.set_index("path").session_start
gap = pd.Series([abs(pd.Timestamp(st[a]) - pd.Timestamp(st[b])).total_seconds()
                 for a, b in zip(dropped.kept, dropped.dropped)])
print("session_start_time difference within a duplicate pair (s): median", gap.median(), "max", gap.max())
print(dropped.head(6).to_string(index=False))

Most recordings are deposited twice, under two session start times a few seconds to a minute apart, with
identical spike times and trial starts. Each recording is counted once from here on.

In [ ]:
k = kept.copy()
print("Distinct recordings by recorded area and tFUS target (from general/notes):")
print(pd.crosstab(k.area.replace("", "(empty)"), k.stim.replace("", "(empty)"), margins=True).to_string())
print("\nRats:", k.subject_id.nunique(), sorted(k.subject_id.unique()))
groups = k.groupby("subject_id").area.agg(lambda a: ",".join(sorted(set(a) - {""})))
print("\nAreas recorded per rat:")
print(groups.value_counts().to_string())
nodc = k.prf_dc.isna() | k.pd_dc.isna()
print("\nRecordings without a PRF or PD in general/data_collection:", int(nodc.sum()), "of", len(k), "; by rat:",
      k[nodc].subject_id.value_counts().to_dict())
both = ~nodc & k.prf_id.notna()
agree = (k.prf_dc[both] == k.prf_id[both]) & (np.isclose(k.pd_dc[both], k.pd_id[both]))
print("Recordings where data_collection and the identifier both give PRF and PD:", int(both.sum()),
      "; they agree in", int(agree.sum()))
print(k[both][~agree][["identifier", "data_collection"]].to_string(index=False))
print("Recordings with no PRF or PD in either field:", int((k.prf.isna() | k.pd_us.isna()).sum()),
      k[k.prf.isna() | k.pd_us.isna()].identifier.tolist())
print("\nIdentifier suffixes after <rat>_<PRF>_<PD>_<UD>_<n>:")
print(k.identifier.str.split("_").str[5:].str.join("_").replace("", "(none)").value_counts().to_string())
print("\nAny identifier, note or data_collection naming sham, no tFUS, CG1 or cingulate:",
      int(k[["identifier", "notes", "data_collection"]].apply(
          lambda c: c.str.contains("(?i)sham|cg1|cingul|no tfus|control")).any(axis=1).sum()))

The paper used 16 rats with S1 and POm probes and 12 with S1 and AC probes (Methods, Animal preparation).
The dataset has 20 rats: 10 with POm recordings and 7 with AC recordings; of the other three, BH494 has S1
and M1, BH500 has S1, M1 and recordings whose area is left empty, and BH594 has a single S1 file. It also
holds recordings of M1, and sessions with tFUS aimed at M1 or at POm, which the paper does not describe.
The PRF and PD are taken from `general/data_collection`, and from the identifier where that field is empty
(the counts above say where, and whether the two agree where both are given). None is marked as sham or
as aimed at CG1, so the sham and control sessions of Figure 5 cannot be found.

In [ ]:
k["n_trials"] = k.n_trials.astype(int)
print("Trials per recording:", k.n_trials.describe()[["min", "50%", "max"]].to_dict(),
      "; recordings with 500 trials:", int((k.n_trials == 500).sum()), "of", len(k))
s1 = k[(k.stim == "S1") & (k.pd_us == 200) & k.area.isin(["S1", "POM", "AC"])].copy()
print("\nPD 200 us recordings with tFUS aimed at S1, by area and PRF (Hz):")
print(pd.crosstab(s1.area, s1.prf).to_string())
t0 = time.time()
res = tfus.map_sessions(tfus.session_counts, list(s1.asset_id))
s1["res"] = res
print(f"\nloaded spike times of {len(s1)} recordings in {time.time() - t0:.0f} s")
iti = np.concatenate([np.diff(r["start"]) for r in res])
print("Interval between trial starts (s): min %.3f, median %.3f, max %.3f; paper 2.5 s with 10%% jitter"
      % (iti.min(), np.median(iti), iti.max()))
dur = np.concatenate([np.asarray(tfus.load_session(s1.asset_id.iloc[0])["stop"]) - s1.res.iloc[0]["start"]])
print("Trial stop minus start in one recording (s):", np.unique(np.round(dur, 3)))

In [ ]:
# Are the two areas of a session recorded together? Compare the trial starts of files that share an identifier.
rows = []
for ident, g in s1.groupby("identifier"):
    if len(g) < 2:
        continue
    starts = [r["start"] for r in g.res]
    a, b = starts[0], starts[1]
    same_n = len(a) == len(b)
    rows.append(dict(identifier=ident, areas="+".join(g.area), n=(len(a), len(b)),
                     same_times=bool(same_n and np.allclose(a, b, atol=1e-3)),
                     same_intervals=bool(same_n and np.allclose(np.diff(a), np.diff(b), atol=1e-3))))
pairs = pd.DataFrame(rows)
print(f"{len(pairs)} identifiers with two areas; same trial start times: {pairs.same_times.sum()}; "
      f"same intervals between trials: {pairs.same_intervals.sum()}")
dstart = []
for ident, g in s1.groupby("identifier"):
    if len(g) == 2:
        a, b = (pd.Timestamp(x) for x in g.session_start)
        dstart.append(abs(a - b).total_seconds() / 60)
print("session_start_time difference between the two area files of a session (min): median %.1f, max %.1f"
      % (np.median(dstart), np.max(dstart)))
print(pairs[~pairs.same_intervals].head(10).to_string(index=False))

### Clock check

`start_time` of the trials table is taken as sonication onset. If it is, the units' firing should change
at fixed delays after it; with the trial starts shifted by 1 s (inside the 2.25 s minimum interval, where
no tFUS is delivered), the same measure should show no change. The measure is the mean over units of the
normalized rate (reading N1) in each 67 ms window, for every unit with more than 2000 spikes.

In [ ]:
def unit_table(frame, min_spikes=tfus.MIN_SPIKES):
    rows = []
    for _, r in frame.iterrows():
        res = r["res"]
        for c, cs, ps in zip(res["true"], res["shifted"], res["psth"]):
            if c["n_spikes"] <= min_spikes:
                continue
            rows.append(dict(subject=r.subject_id, identifier=r.identifier, area=r.area, prf=r.prf,
                             true=c, shifted=cs, psth=ps))
    return pd.DataFrame(rows)


units = unit_table(s1)
print("Units with more than 2000 spikes, PD 200 us, tFUS at S1:")
print(pd.crosstab(units.area, units.prf, margins=True).to_string())
rows = []
for (area, prf), g in units.groupby(["area", "prf"]):
    for align in ["true", "shifted"]:
        vals = {w: np.nanmean([tfus.normalized_rates(c, w, "N1") for c in g[align]]) for w in tfus.WINDOWS}
        rows.append(dict(area=area, prf=prf, onsets=align, n=len(g), **{k: round(v, 1) for k, v in vals.items()}))
clock = pd.DataFrame(rows)
print("\nMean normalized rate (%) by window, at the trial starts and 1 s later:")
print(clock.to_string(index=False))
dev = clock.assign(dev=clock[list(tfus.WINDOWS)].sub(100).abs().max(axis=1)).groupby("onsets").dev.median()
print("\nMedian over area and PRF of the largest departure from 100%:", dev.round(1).to_dict())

In [ ]:
centers = (tfus.PSTH_EDGES[:-1] + tfus.PSTH_EDGES[1:]) / 2 * 1000
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharex=True)
for ax, area in zip(axes, ["S1", "POM", "AC"]):
    for prf, g in units[units.area == area].groupby("prf"):
        m = np.mean(np.vstack(g.psth), axis=0) / 0.01
        ax.plot(centers, m, label=f"{int(prf)} Hz (n={len(g)})")
        print(f"{area} {int(prf)} Hz: mean rate {m.min():.1f} to {m.max():.1f} spikes/s; "
              f"[-500, 0) ms mean {m[centers < 0].mean():.1f}, [0, 300) ms max {m[(centers > 0) & (centers < 300)].max():.1f}")
    ax.axvspan(0, 67, color="0.9")
    ax.set_title(f"{area}, PD 200 us")
    ax.set_xlabel("time from trial start (ms)")
axes[0].set_ylabel("mean rate (spikes/s)")
axes[-1].legend(fontsize=7)
plt.tight_layout()
plt.show()

The rates change after the trial start and not 1 s later (the table above gives both), so the trial start
times share the clock of the spike times and mark the stimulus, as the dataset's description says
("intervals contains each tFUS trial start and end timestamps"). That `start_time` is the onset of
sonication itself, and not a trigger at a fixed delay, is consistent with the data (the firing changes
within the first 67 ms window at the high PRFs) but not shown. The intervals between trial starts match
the paper's 2.5 s with 10% jitter. In every session with files from two areas, both files carry the same
trial start times, so the two areas were recorded together on one clock (the description says "Each
recording contains two areas"), though their `session_start_time` values differ (by the minutes printed
above), so that field does not give the start of the recording.

### Unit totals for the count claims

`fig4-unit-counts` and `fig6-unit-counts` give RSU and FSU counts, which need the cell types. The totals
per area (units with more than 2000 spikes, distinct recordings, tFUS at S1, PD 200 us) are shown here
to say what the dataset holds; they decide no outcome. S1 is counted two ways: every S1 recording, and only
the S1 recordings whose identifier also has a POm recording (the S1 and POm rats of Figure 4).

In [ ]:
def n_above(row, min_spikes=tfus.MIN_SPIKES):
    return int(sum(n > min_spikes for n in json.loads(row)))


k["n_good"] = k.n_spikes.map(n_above)
pd200 = k[(k.stim == "S1") & (k.pd_us == 200)]
pom_ids = set(pd200[pd200.area == "POM"].identifier)
pom_rats = set(pd200[pd200.area == "POM"].subject_id)
tab = []
for prf in [30, 300, 1500, 3000, 4500]:
    g = pd200[pd200.prf == prf]
    tab.append(dict(prf=prf,
                    S1_all=int(g[g.area == "S1"].n_good.sum()),
                    S1_in_POm_rats=int(g[(g.area == "S1") & g.subject_id.isin(pom_rats)].n_good.sum()),
                    S1_paired_with_POm=int(g[(g.area == "S1") & g.identifier.isin(pom_ids)].n_good.sum()),
                    POm=int(g[g.area == "POM"].n_good.sum()), AC=int(g[g.area == "AC"].n_good.sum())))
tab = pd.DataFrame(tab).set_index("prf")
tab.loc["all PRFs"] = tab.sum()
print(tab.to_string())
print("\nPaper (Fig. 4 legend), RSU + FSU: S1 3000 Hz 366, POm 3000 Hz 250, S1 30 Hz 209, POm 30 Hz 232;"
      " Fig. 6 legend: S1 1297, POm 1019 over the five PRFs; Fig. 11: AC 453.")

### The STTC files

The 370 files under `sub-STTC-*` hold STTC values between the units of two areas. They are listed here
because the claims file names them as a variant for Figure 10; they carry no unit identities or cell
types.

In [ ]:
sttc = inv[inv.kind == "STTC"]
print(sttc.notes.value_counts().to_string())
import h5py, remfile
f = h5py.File(remfile.File(tfus.asset_url(sttc.asset_id.iloc[0])), "r")
print("\n", sttc.path.iloc[0], f["identifier"][()].decode(), f["general/notes"][()].decode())
print(f["units"].attrs["description"])
for c in ["STTC", "STTCAtoB", "STTCBtoA", "STTCA", "STTCB"]:
    print(c, f["units"][c].shape, "|", f["units"][c].attrs["description"])

The STTC arrays do not have the shape their descriptions give: the descriptions list a dimension for the
nine values of dt, while the arrays have the 10 time windows, the trials and the units of the two areas,
and no dt dimension. Which dt the values were computed with is not stated, and no unit is typed, so the
stored values cannot stand in for the STTC between S1 RSUs and POm RSUs of Figure 10.

## Untyped responses in S1 and POm (a variant; decides no outcome)

The claims on S1 and POm need the RSU/FSU split. For orientation only, the same measure on all units of
each area, without typing (reading N1, comparison B1, Bonferroni over the five PRFs):

In [ ]:
rows = []
for area in ["S1", "POM"]:
    for prf, g in units[units.area == area].groupby("prf"):
        base = tfus.baseline_sample(list(g["true"]), "N1", "B1")
        r = dict(area=area, prf=int(prf), n=len(g))
        for w in tfus.WINDOWS:
            x = np.array([tfus.normalized_rates(c, w, "N1") for c in g["true"]])
            p_up, _ = tfus.vs_baseline(x, base, 5, "greater")
            p_dn, _ = tfus.vs_baseline(x, base, 5, "less")
            r[w] = f"{np.nanmean(x):.0f}{'+' if p_up < 0.01 else ('-' if p_dn < 0.01 else '')}"
        rows.append(r)
print("Mean normalized rate (%) of all units; + above, - below baseline at corrected p < 0.01:")
print(pd.DataFrame(rows).to_string(index=False))

## fig11b-ac-responses

> AC neurons in the S1-AC rats show time-locked responses to 1500 and 3000 Hz PRF and postsonication
> responses to 30, 1500, 3000 and 4500 Hz, both PRF dependent, least at 300 Hz.

Criterion (claims file): the header's spiking method on all AC units with more than 2000 spikes in the PRF
series with 200 us PD; windows 0 to 67 and 201 to 268 ms (the 67 to 134 and 134 to 201 ms bins shown).
Reproduced when, under every method reading, both Kruskal-Wallis tests have p < 0.01, exactly 1500 and
3000 Hz are above baseline during sonication, exactly 30, 1500, 3000 and 4500 Hz are above baseline
postsonication, and 300 Hz has the lowest mean normalized rate in at least one window; partially
reproduced when the sets differ by one PRF in one window or the 300 Hz part fails. Expected: 453 AC
neurons over the five PRFs.

The method readings (claims file header): N1 or N2 for the normalized rate; B1 or B2 for the baseline
sample; the Bonferroni factor is the five PRF levels of the panel (C1; C2 applies only to a panel with a
single comparison, so it is the same here). B3, a paired signed-rank test, is shown as a variant.

In [ ]:
ac = units[units.area == "AC"].copy()
PRFS = [30.0, 300.0, 1500.0, 3000.0, 4500.0]
print("AC units with more than 2000 spikes, by rat and PRF:")
print(pd.crosstab(ac.subject, ac.prf, margins=True).to_string())
print(f"\nTotal {len(ac)} (expected 453 +/- 45.3)")
allac = s1[s1.area == "AC"]
print("AC recordings used:", len(allac), "; units in them before the 2000-spike rule:",
      int(sum(r["n_units"] for r in allac.res)))

In [ ]:
def evaluate(frame, nreading, breading, min_label=""):
    out, sig = [], {}
    for w in tfus.WINDOWS:
        groups, line = [], {}
        for p in PRFS:
            cs = list(frame[frame.prf == p]["true"])
            x = np.array([tfus.normalized_rates(c, w, nreading) for c in cs])
            groups.append(x)
            if breading == "B3":
                from scipy import stats as st
                win = np.array([c[w].mean() for c in cs]); bas = np.array([c["base"].mean() for c in cs])
                pr = st.wilcoxon(win, bas, alternative="greater").pvalue
                pc = min(1, pr * 5)
            else:
                base = tfus.baseline_sample(cs, nreading, breading)
                pc, pr = tfus.vs_baseline(x, base, 5, "greater")
            sem = np.nanstd(x, ddof=1) / np.sqrt(np.isfinite(x).sum())
            out.append(dict(N=nreading, B=breading, window=w, prf=int(p), n=int(np.isfinite(x).sum()),
                            mean=round(np.nanmean(x), 1), sem=round(sem, 1), median=round(np.nanmedian(x), 1),
                            p_corrected=float(f"{pc:.3g}"), above=pc < 0.01))
        kw = tfus.kruskal_levels(groups)
        for o in out[-5:]:
            o["kruskal_p"] = float(f"{kw:.3g}")
    return pd.DataFrame(out)


def judge(t):
    def sset(w):
        return {r.prf for r in t[(t.window == w) & t.above].itertuples()}
    during, post = sset("0-67"), sset("201-268")
    kw_ok = (t[t.window == "0-67"].kruskal_p.iloc[0] < 0.01) and (t[t.window == "201-268"].kruskal_p.iloc[0] < 0.01)
    d_err = len(during ^ {1500, 3000})
    p_err = len(post ^ {30, 1500, 3000, 4500})
    low = [t[t.window == w].sort_values("mean").prf.iloc[0] for w in ["0-67", "201-268"]]
    low300 = 300 in low
    if kw_ok and d_err == 0 and p_err == 0 and low300:
        verdict = "reproduced"
    elif kw_ok and ((d_err + p_err == 1 and low300) or (d_err == 0 and p_err == 0 and not low300)):
        verdict = "partially reproduced"
    else:
        verdict = "not reproduced"
    return dict(kruskal_both=kw_ok, during=sorted(during), post=sorted(post), lowest_0_67=low[0],
                lowest_201_268=low[1], verdict=verdict)


tables, verdicts = {}, []
for N in ["N1", "N2"]:
    for B in ["B1", "B2"]:
        t = evaluate(ac, N, B)
        tables[(N, B)] = t
        verdicts.append(dict(N=N, B=B, **judge(t)))
t_b3 = evaluate(ac, "N1", "B3")
verdicts.append(dict(N="raw rates", B="B3 (variant)", **judge(t_b3)))
print("Outcome under each reading (sets are the PRFs above baseline at corrected p < 0.01):")
print(pd.DataFrame(verdicts).to_string(index=False))

In [ ]:
for (N, B), t in tables.items():
    print(f"\nReading {N}, {B}: mean, SEM and median normalized rate (%), corrected p against baseline, Kruskal-Wallis p")
    print(t.pivot_table(index="prf", columns="window", values=["mean", "p_corrected"], sort=False)
          .reindex(columns=list(tfus.WINDOWS), level=1).to_string())
    print("Kruskal-Wallis p by window:", t.groupby("window", sort=False).kruskal_p.first().to_dict())
print("\nVariant B3 (paired signed-rank of raw rates against [-67, 0) ms), corrected p:")
print(t_b3.pivot_table(index="prf", columns="window", values="p_corrected", sort=False).to_string())

The two normalizations differ in kind. N2 averages per-trial ratios over the trials with a nonzero
baseline count; at these firing rates most 67 ms baselines hold no spike, and keeping only the trials with
one makes the baseline larger than average, so N2 sits below 100% even where nothing happens. The next
cell measures this at the trial starts shifted by 1 s, where there is no stimulus.

In [ ]:
rows = []
for N in ["N1", "N2"]:
    for w in tfus.WINDOWS:
        x = np.array([tfus.normalized_rates(c, w, N) for c in ac["shifted"]])
        rows.append(dict(N=N, window=w, mean_at_shifted_onsets=round(np.nanmean(x), 1),
                         median=round(np.nanmedian(x), 1)))
print("AC units, normalized rate (%) with trial starts shifted by 1 s (no stimulus):")
print(pd.DataFrame(rows).pivot(index="N", columns="window", values="mean_at_shifted_onsets")
      [list(tfus.WINDOWS)].to_string())
nz = np.array([np.mean(c["base"] > 0) for c in ac["true"]])
print(f"\nFraction of trials with a nonzero [-67, 0) ms count, over AC units: median {np.median(nz):.2f}, "
      f"range {nz.min():.2f} to {nz.max():.2f}")

In [ ]:
# By rat: mean normalized rate (N1) in the two windows of the criterion
rows = []
for (sub, prf), g in ac.groupby(["subject", "prf"]):
    r = dict(rat=sub, prf=int(prf), n=len(g))
    for w in ["0-67", "201-268"]:
        r[w] = round(np.nanmean([tfus.normalized_rates(c, w, "N1") for c in g["true"]]), 1)
    rows.append(r)
print("Mean normalized rate (%) by rat, reading N1:")
print(pd.DataFrame(rows).to_string(index=False))

# The 2000-spike rule: how the count and the result change with it
rows = []
for m in [1000, 2000, 4000, 8000]:
    sub = unit_table(s1[s1.area == "AC"], min_spikes=m)
    j = judge(evaluate(sub, "N1", "B1"))
    rows.append(dict(min_spikes=m, n_units=len(sub), during=j["during"], post=j["post"],
                     lowest=(j["lowest_0_67"], j["lowest_201_268"]), verdict=j["verdict"]))
print("\nReading N1, B1 under other spike-count rules (the paper's is more than 2000):")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
t = tables[("N1", "B1")]
fig, ax = plt.subplots(figsize=(7, 3.5))
for p in PRFS:
    q = t[t.prf == int(p)]
    ax.errorbar(range(4), q["mean"], yerr=q["sem"], marker="o", label=f"{int(p)} Hz", capsize=3)
ax.axhline(100, color="0.5", lw=0.8)
ax.set_xticks(range(4))
ax.set_xticklabels([f"{w} ms" for w in tfus.WINDOWS])
ax.set_ylabel("normalized rate (%), mean +/- SEM")
ax.set_title("AC units, PD 200 us, reading N1")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
print("Plotted means range from", t["mean"].min(), "to", t["mean"].max(), "%; none outside the axes.")

**Result: partially reproduced, resting on the assumption of reading N1 with baseline B1.** Under N1 and
B1 both Kruskal-Wallis tests have p < 0.01 (7.83e-06 during sonication, 5.16e-49 at 201 to 268 ms), the
postsonication set is exactly 30, 1500, 3000 and 4500 Hz, and 300 Hz has the lowest mean in both windows
(99.7% and 94.3%), but during sonication 4500 Hz is also above baseline (108.0%, corrected p = 7.05e-05)
besides 1500 Hz (140.3%) and 3000 Hz (107.1%), so the time-locked set differs by one PRF. The readings
disagree: with B2 the time-locked set also gains 300 Hz (not reproduced), and under N2 the Kruskal-Wallis
test during sonication has p = 0.0213 and at most one PRF per window is above baseline (not reproduced).
N2 gives 49.8% to 61.7% with no stimulus (trial starts shifted by 1 s), where N1 gives 94.3% to 99.0%, so N2
cannot give a rate normalized to the 67 ms before stimulation, and N1 with the paper's own pre-stimulus
window as the baseline (B1) is adopted. The variant B3 gives the same sets as N1 with B1. The result holds
for spike-count rules from 1000 to 8000 spikes. The dataset gives 633 AC units with more than 2000 spikes
(expected 453 +/- 45.3), from 6 rats, more than the paper at every PRF.

## Summary

| Claim | Outcome |
|---|---|
| fig11b-ac-responses | Partially reproduced (assumed: normalization N1, baseline B1) |
| fig4-unit-counts, fig6-unit-counts | Blocked: data missing (cell types) |
| fig4a, fig4b, fig4-postsonication-3000, fig4-30hz-responses, fig4-fsu-no-change | Blocked: data missing (cell types) |
| fig5a-sham-no-change, fig5b-cg1-control-no-change | Blocked: data missing (cell types; no sham or CG1 sessions marked) |
| fig6a (2 claims), fig6b, fig6c, fig6d | Blocked: data missing (cell types) |
| fig7a to fig7d, fig8a, fig8b, fig9a, fig9b | Blocked: data missing (cell types) |
| fig10a to fig10f | Blocked: data missing (cell types) |
| pathway-follows-local-prf | Blocked: depends on blocked claims |
| fig11c-ac-to-s1-30hz, fig11d-ac-to-s1-3000 | Blocked: data missing (S1 cell types) |
| auditory-confound-low-prf | Blocked: depends on blocked claims |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's: the normalization and the baseline sample are not fixed by the
paper; reading N1 with baseline B1 is adopted and the others are shown. Duplicate files are counted once.

What was found about the dataset: no spike waveforms and no cell types (`celltype_label` is 0 for every
unit, though described as 1 = RSU and 2 = FSU), so 32 of the 33 claims cannot be tested; most recordings
are deposited twice; the sham and CG1 sessions cannot be identified; the dataset has 10 rats with POm
recordings and 7 with AC recordings, where the paper reports 16 and 12, together with M1 recordings and
tFUS aimed at M1 or POm that the paper does not describe; `session_start_time` differs between the two area
files of one session; the STTC files lack the dt dimension their descriptions give and identify no units.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")